In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.utils import to_categorical


# ============================================================
# LOAD MNIST DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = mnist.load_data()


# ============================================================
# PREPROCESSING
# ============================================================

# Add channel dimension
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Convert grayscale images to RGB
x_train = tf.image.grayscale_to_rgb(
    tf.cast(x_train, tf.float32)
)

x_test = tf.image.grayscale_to_rgb(
    tf.cast(x_test, tf.float32)
)

# One-hot encoding
y_train = to_categorical(
    y_train,
    10
)

y_test = to_categorical(
    y_test,
    10
)


# ============================================================
# CREATE DATASETS
# ============================================================

# Do NOT resize the complete dataset at once.
# Resize images batch-by-batch.

train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)


def preprocess_image(image, label):

    # Resize one image
    image = tf.image.resize(
        image,
        (224, 224)
    )

    # VGG19 preprocessing
    image = preprocess_input(
        image
    )

    return image, label


train_dataset = train_dataset.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)


# Batch the data
train_dataset = train_dataset.batch(
    32
).prefetch(
    tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(
    32
).prefetch(
    tf.data.AUTOTUNE
)


# ============================================================
# MODEL 1
# ============================================================

base_model1 = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained VGG19 layers
base_model1.trainable = False


model1 = Sequential([
    base_model1,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation="relu"
    ),

    Dense(
        10,
        activation="softmax"
    )
])


model1.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


model1.fit(
    train_dataset,
    epochs=5,
    verbose=1
)


# Evaluate Model 1

loss1, accuracy1 = model1.evaluate(
    test_dataset,
    verbose=1
)


# ============================================================
# MODEL 2
# ============================================================

base_model2 = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained VGG19 layers
base_model2.trainable = False


model2 = Sequential([
    base_model2,

    GlobalAveragePooling2D(),

    Dense(
        256,
        activation="relu"
    ),

    Dense(
        10,
        activation="softmax"
    )
])


model2.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


model2.fit(
    train_dataset,
    epochs=5,
    verbose=1
)


# Evaluate Model 2

loss2, accuracy2 = model2.evaluate(
    test_dataset,
    verbose=1
)


# ============================================================
# PREDICTIONS
# ============================================================

# Prepare only the first 2 images for prediction

prediction_images = x_test[:2]

prediction_images = tf.image.resize(
    prediction_images,
    (224, 224)
)

prediction_images = preprocess_input(
    prediction_images
)


pred1 = model1.predict(
    prediction_images,
    verbose=0
)

pred2 = model2.predict(
    prediction_images,
    verbose=0
)


# ============================================================
# MODEL 1 RESULTS
# ============================================================

print("\n==============================")
print("MODEL 1 RESULTS")
print("==============================")

print(
    "Accuracy:",
    round(accuracy1 * 100, 2),
    "%"
)

print(
    "Loss:",
    round(loss1, 4)
)


plt.figure(figsize=(8, 4))


for i in range(2):

    predicted = pred1[i].argmax()

    actual = y_test[i].argmax()

    vector = y_test[i].astype(int)

    vector_text = "[" + ",".join(
        map(str, vector)
    ) + "]"


    plt.subplot(
        1,
        2,
        i + 1
    )

    plt.imshow(
        x_test[i].numpy().reshape(28, 28),
        cmap="gray"
    )

    plt.title(
        f"Predicted: {predicted}\n"
        f"Actual: {actual}\n"
        f"One-Hot: {vector_text}"
    )

    plt.axis("off")


plt.suptitle(
    "Model 1 Predictions"
)

plt.tight_layout()

plt.show()


# ============================================================
# MODEL 2 RESULTS
# ============================================================

print("\n==============================")
print("MODEL 2 RESULTS")
print("==============================")

print(
    "Accuracy:",
    round(accuracy2 * 100, 2),
    "%"
)

print(
    "Loss:",
    round(loss2, 4)
)


plt.figure(figsize=(8, 4))


for i in range(2):

    predicted = pred2[i].argmax()

    actual = y_test[i].argmax()

    vector = y_test[i].astype(int)

    vector_text = "[" + ",".join(
        map(str, vector)
    ) + "]"


    plt.subplot(
        1,
        2,
        i + 1
    )

    plt.imshow(
        x_test[i].numpy().reshape(28, 28),
        cmap="gray"
    )

    plt.title(
        f"Predicted: {predicted}\n"
        f"Actual: {actual}\n"
        f"One-Hot: {vector_text}"
    )

    plt.axis("off")


plt.suptitle(
    "Model 2 Predictions"
)

plt.tight_layout()

plt.show()




Epoch 1/5


 319/1875 [====>.........................] - ETA: 58:49 - loss: 0.3670 - accuracy: 0.8915